In [ ]:
import os
import gc
import json
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, Features, Value
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)


In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"






SEED = 42

MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"


SAMPLED_DATA_PATH = "./digikala_1200k_cluster_sample_rate.csv"
VALIDATION_TEST_PATH = "./test_not_mismatch.csv"

MAX_LENGTH = 128

NUM_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 3
EVAL_STEPS = 10000

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
ACCUM_STEPS = 1

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
LABEL_SMOOTHING = 0.1


REMOVE_LEAKAGE = True

FILTER_BY_RATE = False

HIGH_RATE = 4.5
LOW_RATE = 1.5
POSITIVE_LABEL = "recommended"
NEGATIVE_LABEL = "not_recommended"


TRAIN_SUBSET = None

OUTPUT_DIR = "./parsbert_digikala_checkpoints_label_clean_KMEANS"
FINAL_MODEL_DIR = "./parsbert_digikala_best_label_clean_KMEANS"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

set_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


if not os.path.exists(SAMPLED_DATA_PATH):
    raise FileNotFoundError(
        f"Training dataset not found: {SAMPLED_DATA_PATH}"
    )

train_columns = ["text", "recommendation_status_clean"]
if FILTER_BY_RATE:
    train_columns.append("rate")

balanced_data = pd.read_csv(
    SAMPLED_DATA_PATH,
    usecols=train_columns,
    encoding="utf-8-sig",
    low_memory=False
)

print("\nTraining dataset loaded:", len(balanced_data))


balanced_data["text"] = (
    balanced_data["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

balanced_data["recommendation_status_clean"] = (
    balanced_data["recommendation_status_clean"]
    .astype("string")
    .str.strip()
)

balanced_data = balanced_data[
    balanced_data["text"].ne("") &
    balanced_data["recommendation_status_clean"].notna() &
    balanced_data["recommendation_status_clean"].ne("")
].copy()

print("\nTrain class distribution:")
print(
    balanced_data["recommendation_status_clean"].value_counts()
)


if not os.path.exists(VALIDATION_TEST_PATH):
    raise FileNotFoundError(
        f"Validation/Test dataset not found: {VALIDATION_TEST_PATH}"
    )

validation_test_df = pd.read_csv(
    VALIDATION_TEST_PATH,
    usecols=["text", "recommendation_status"],
    encoding="utf-8-sig",
    low_memory=False
)

validation_test_df["text"] = (
    validation_test_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

validation_test_df["recommendation_status"] = (
    validation_test_df["recommendation_status"]
    .astype("string")
    .str.strip()
)

validation_test_df = validation_test_df[
    validation_test_df["text"].ne("") &
    validation_test_df["recommendation_status"].notna() &
    validation_test_df["recommendation_status"].ne("")
].copy()

validation_test_df = (
    validation_test_df
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print("\nValidation/Test size:", len(validation_test_df))

print("\nValidation/Test distribution:")
print(
    validation_test_df["recommendation_status"].value_counts()
)


leaked = balanced_data["text"].isin(set(validation_test_df["text"]))
print(f"\nTrain rows that also appear in val/test: {int(leaked.sum()):,}")

if REMOVE_LEAKAGE and leaked.any():
    balanced_data = balanced_data[~leaked].copy()
    print("Removed leaked rows. Train size now:", f"{len(balanced_data):,}")


if FILTER_BY_RATE:
    rate = pd.to_numeric(balanced_data["rate"], errors="coerce")
    label = balanced_data["recommendation_status_clean"]

    contradict = (rate > 0) & (
        ((rate >= HIGH_RATE) & (label == NEGATIVE_LABEL)) |
        ((rate <= LOW_RATE) & (label == POSITIVE_LABEL))
    )

    print(f"Rows contradicting rate: {int(contradict.sum()):,}")
    balanced_data = balanced_data[~contradict].copy()
    print("Train size after rate filter:", f"{len(balanced_data):,}")

    balanced_data = balanced_data.drop(columns=["rate"])


if TRAIN_SUBSET is not None and TRAIN_SUBSET < len(balanced_data):
    balanced_data = balanced_data.sample(
        n=TRAIN_SUBSET,
        random_state=SEED
    ).copy()
    print("Using a subset of the train set:", f"{len(balanced_data):,}")

balanced_data = balanced_data.reset_index(drop=True)


train_labels = set(
    balanced_data["recommendation_status_clean"].astype(str).unique()
)

eval_labels = set(
    validation_test_df["recommendation_status"].astype(str).unique()
)

print("\nTrain labels:", train_labels)
print("Validation/Test labels:", eval_labels)

if train_labels != eval_labels:
    raise ValueError(
        "Train and Validation/Test labels are not identical.\n"
        f"Train-only: {train_labels - eval_labels}\n"
        f"Eval-only: {eval_labels - train_labels}"
    )

print("\nLabel consistency check passed.")


label_encoder = LabelEncoder()

balanced_data = balanced_data.copy()

balanced_data["labels"] = label_encoder.fit_transform(
    balanced_data["recommendation_status_clean"].astype(str)
)

id2label = {
    int(i): str(label)
    for i, label in enumerate(label_encoder.classes_)
}

label2id = {
    str(label): int(i)
    for i, label in enumerate(label_encoder.classes_)
}

print("\nLabel mapping:", id2label)


val_df, test_df = train_test_split(
    validation_test_df,
    test_size=0.50,
    random_state=SEED,
    stratify=validation_test_df["recommendation_status"]
)

val_df = val_df.copy()
test_df = test_df.copy()

val_df["labels"] = label_encoder.transform(
    val_df["recommendation_status"].astype(str)
)

test_df["labels"] = label_encoder.transform(
    test_df["recommendation_status"].astype(str)
)

print("\nTrain size:", len(balanced_data))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))


print("\nTrain distribution:")
print(
    balanced_data["recommendation_status_clean"].value_counts()
)

print("\nValidation distribution:")
print(
    val_df["recommendation_status"].value_counts()
)

print("\nTest distribution:")
print(
    test_df["recommendation_status"].value_counts()
)


gc.collect()

dataset_features = Features({
    "text": Value("string"),
    "labels": Value("int64")
})


def dataframe_generator(texts, labels):

    for text, label in zip(texts, labels):

        if pd.isna(text):
            text = ""
        else:
            text = str(text)

        yield {
            "text": text,
            "labels": int(label)
        }


print("\nCreating Train Dataset...")

train_ds = Dataset.from_generator(
    dataframe_generator,
    gen_kwargs={
        "texts": balanced_data["text"],
        "labels": balanced_data["labels"]
    },
    features=dataset_features,
    writer_batch_size=500,
    keep_in_memory=False
)


print("\nCreating Validation Dataset...")

val_ds = Dataset.from_generator(
    dataframe_generator,
    gen_kwargs={
        "texts": val_df["text"],
        "labels": val_df["labels"]
    },
    features=dataset_features,
    writer_batch_size=500,
    keep_in_memory=False
)


print("\nCreating Test Dataset...")

test_ds = Dataset.from_generator(
    dataframe_generator,
    gen_kwargs={
        "texts": test_df["text"],
        "labels": test_df["labels"]
    },
    features=dataset_features,
    writer_batch_size=500,
    keep_in_memory=False
)


print("\nDataset creation completed.")

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("\nTokenizer loaded.")


def tokenize_function(batch):

    texts = [
        str(text) if text is not None else ""
        for text in batch["text"]
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH
    )


print("\nTokenizing training dataset...")

train_ds = train_ds.map(
    tokenize_function,
    batched=True,
    batch_size=128,
    writer_batch_size=500,
    remove_columns=["text"]
)


print("\nTokenizing validation dataset...")

val_ds = val_ds.map(
    tokenize_function,
    batched=True,
    batch_size=128,
    writer_batch_size=500,
    remove_columns=["text"]
)


print("\nTokenizing test dataset...")

test_ds = test_ds.map(
    tokenize_function,
    batched=True,
    batch_size=128,
    writer_batch_size=500,
    remove_columns=["text"]
)


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

gc.collect()

print("\nTokenization completed successfully.")


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

print("\nClassification model loaded.")


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }


use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)


steps_per_epoch = -(-len(train_ds) // (TRAIN_BATCH_SIZE * ACCUM_STEPS))
total_steps = steps_per_epoch * NUM_EPOCHS
WARMUP_STEPS = max(1, int(WARMUP_RATIO * total_steps))
print(f"Steps/epoch: {steps_per_epoch:,} | total: {total_steps:,} | warmup: {WARMUP_STEPS:,}")

training_kwargs = dict(

    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    label_smoothing_factor=LABEL_SMOOTHING,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,


    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,

    metric_for_best_model="eval_accuracy",
    greater_is_better=True,

    load_best_model_at_end=True,

    save_total_limit=2,


    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,

    logging_steps=500,

    report_to="none",

    seed=SEED
)



_valid_args = set(inspect.signature(TrainingArguments.__init__).parameters)

if "dataloader_num_workers" in _valid_args:
    training_kwargs["dataloader_num_workers"] = 2


if "group_by_length" in _valid_args:
    training_kwargs["group_by_length"] = True
    print("Speed-up enabled: group_by_length")
elif "train_sampling_strategy" in _valid_args:
    training_kwargs["train_sampling_strategy"] = "group_by_length"
    print("Speed-up enabled: train_sampling_strategy=group_by_length")
else:
    print("Length-grouped batching not available in this version; skipping.")

training_args = TrainingArguments(**training_kwargs)


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=0.0
        )
    ]
)


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nStarting ParsBERT training...")

trainer.train()


print(
    "\nBest checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation accuracy:",
    trainer.state.best_metric
)

print(
    "Best global step:",
    trainer.state.best_global_step
)


test_output = trainer.predict(test_ds)

y_true = test_output.label_ids

y_pred = np.argmax(
    test_output.predictions,
    axis=-1
)

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\nTest Accuracy:", round(test_accuracy, 4))
print("Test Macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 4))

print("\nTest Classification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(label_encoder.classes_))),
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


trainer.save_model(FINAL_MODEL_DIR)

tokenizer.save_pretrained(FINAL_MODEL_DIR)

with open(
    os.path.join(
        FINAL_MODEL_DIR,
        "label_mapping.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "id2label": id2label,
            "label2id": label2id,
            "max_length": MAX_LENGTH
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nBest model saved to:", FINAL_MODEL_DIR)
print("Training and evaluation completed successfully.")


In [ ]:
MODEL_DIR = Path(
    "./parsbert_digikala_best_label_clean_KMEANS"
)

TEST_DATA_PATH = "./digikala_synthetic_comments_10000.csv"

OUTPUT_DIR = Path("./parsbert_test_error_analysis_10000")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "text"
LABEL_COLUMN = "my_label"

SEED = 42
BATCH_SIZE = 8

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Saved model not found: {MODEL_DIR}"
    )

mapping_path = MODEL_DIR / "label_mapping.json"

if not mapping_path.exists():
    raise FileNotFoundError(
        f"Label mapping not found: {mapping_path}"
    )

with open(mapping_path, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

label2id = {
    str(label): int(idx)
    for label, idx in label_mapping["label2id"].items()
}

id2label = {
    int(idx): str(label)
    for idx, label in label_mapping["id2label"].items()
}

MAX_LENGTH = int(label_mapping["max_length"])

CLASS_NAMES = [
    id2label[i]
    for i in range(len(id2label))
]

print("\nClasses:", CLASS_NAMES)
print("Max length:", MAX_LENGTH)


if not os.path.exists(TEST_DATA_PATH):
    raise FileNotFoundError(TEST_DATA_PATH)

validation_test_df = pd.read_csv(
    TEST_DATA_PATH,
    usecols=[TEXT_COLUMN, LABEL_COLUMN],
    encoding="utf-8-sig",
    low_memory=False
)

validation_test_df[TEXT_COLUMN] = (
    validation_test_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)

validation_test_df[LABEL_COLUMN] = (
    validation_test_df[LABEL_COLUMN]
    .astype("string")
    .str.strip()
)

validation_test_df = validation_test_df[
    validation_test_df[TEXT_COLUMN].ne("") &
    validation_test_df[LABEL_COLUMN].notna() &
    validation_test_df[LABEL_COLUMN].ne("")
].copy()


validation_test_df = (
    validation_test_df
    .drop_duplicates(subset=[TEXT_COLUMN])
    .reset_index(drop=True)
)

print("\nValidation/Test total:", len(validation_test_df))


val_df, test_df = train_test_split(
    validation_test_df,
    test_size=0.50,
    random_state=SEED,
    stratify=validation_test_df[LABEL_COLUMN]
)

test_df = test_df.copy().reset_index(drop=True)

print("Test size:", len(test_df))

print("\nTest class distribution:")
print(test_df[LABEL_COLUMN].value_counts())


unknown_labels = (
    set(test_df[LABEL_COLUMN].astype(str).unique())
    - set(label2id.keys())
)

if unknown_labels:
    raise ValueError(
        f"Unknown labels in test data: {unknown_labels}"
    )

y_true = np.array([
    label2id[label]
    for label in test_df[LABEL_COLUMN].astype(str)
])


print("\nLoading saved model...")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR)
)

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

model.to(device)
model.eval()

print("Model loaded successfully.")
print("Training will NOT be performed.")


all_probabilities = []

texts = test_df[TEXT_COLUMN].tolist()

print("\nStarting test prediction...")

with torch.inference_mode():

    for start in range(0, len(texts), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(texts))

        batch_texts = texts[start:end]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        if device.type == "cuda":
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**encoded)
        else:
            outputs = model(**encoded)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=-1
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

        if end == len(texts) or end % 1000 == 0:
            print(f"Predicted: {end:,}/{len(texts):,}")

        del encoded, outputs, probabilities

        if device.type == "cuda":
            torch.cuda.empty_cache()


all_probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

y_pred = np.argmax(all_probabilities, axis=1)

predicted_labels = [
    id2label[int(idx)]
    for idx in y_pred
]

true_labels = test_df[LABEL_COLUMN].astype(str).tolist()

confidence = all_probabilities.max(axis=1)

correct = (y_true == y_pred)


results = pd.DataFrame({
    "text": texts,
    "true_label": true_labels,
    "predicted_label": predicted_labels,
    "confidence": confidence,
    "correct": correct
})


for class_id, class_name in id2label.items():

    results[f"prob_{class_name}"] = (
        all_probabilities[:, class_id]
    )

all_predictions_path = (
    OUTPUT_DIR / "all_test_predictions.csv"
)

results.to_csv(
    all_predictions_path,
    index=False,
    encoding="utf-8-sig"
)


errors = results[
    results["correct"] == False
].copy()

errors = errors.sort_values(
    by="confidence",
    ascending=False
).reset_index(drop=True)

errors_path = OUTPUT_DIR / "test_errors.csv"

errors.to_csv(
    errors_path,
    index=False,
    encoding="utf-8-sig"
)


accuracy = accuracy_score(y_true, y_pred)

print("\n" + "=" * 65)
print("TEST RESULTS")
print("=" * 65)

print("Test samples:", len(results))
print("Correct predictions:", int(correct.sum()))
print("Incorrect predictions:", len(errors))
print("Test accuracy:", round(accuracy, 5))
print("Test error rate:", round(1 - accuracy, 5))

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)


cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in CLASS_NAMES],
    columns=[f"pred_{x}" for x in CLASS_NAMES]
)

print("\nConfusion Matrix:")
print(cm_df)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig"
)


print("\nError counts by true class:")

print(
    errors["true_label"].value_counts()
)

print("\nWrong prediction combinations:")

print(
    errors.groupby(
        ["true_label", "predicted_label"]
    ).size().sort_values(ascending=False)
)


summary = {
    "test_samples": int(len(results)),
    "correct_predictions": int(correct.sum()),
    "incorrect_predictions": int(len(errors)),
    "accuracy": float(accuracy),
    "error_rate": float(1 - accuracy),
    "classes": CLASS_NAMES,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH
}

with open(
    OUTPUT_DIR / "summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n" + "=" * 65)
print("ERROR ANALYSIS COMPLETED")
print("=" * 65)

print("\nAll predictions:")
print(all_predictions_path.resolve())

print("\nOnly incorrect predictions:")
print(errors_path.resolve())

print("\nOutput directory:")
print(OUTPUT_DIR.resolve())

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

Classes: ['no_idea', 'not_recommended', 'recommended']
Max length: 128

Validation/Test total: 10000
Test size: 5000

Test class distribution:
my_label
recommended        1667
no_idea            1667
not_recommended    1666
Name: count, dtype: Int64

Loading saved model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.
Training will NOT be performed.

Starting test prediction...
Predicted: 1,000/5,000
Predicted: 2,000/5,000
Predicted: 3,000/5,000
Predicted: 4,000/5,000
Predicted: 5,000/5,000

TEST RESULTS
Test samples: 5000
Correct predictions: 4311
Incorrect predictions: 689
Test accuracy: 0.8622
Test error rate: 0.1378

Classification Report:

                 precision    recall  f1-score   support

        no_idea     0.7132    0.9814    0.8261      1667
not_recommended     1.0000    0.7185    0.8362      1666
    recommended     0.9795    0.8866    0.9307      1667

       accuracy                         0.8622      5000
      macro avg     0.8975    0.8622    0.8643      5000
   weighted avg     0.8975    0.8622    0.8643      5000


Confusion Matrix:
                      pred_no_idea  pred_not_recommended  pred_recommended
true_no_idea                  1636                     0                31
true_not_recommended           469                  1197             

In [ ]:
MODEL_DIR = Path(
    "./parsbert_digikala_best_label_clean_KMEANS"
)

TEST_DATA_PATH = "./test_10k_corrected.csv"

TEXT_COLUMN = "text"
LABEL_COLUMN = "recommendation_status"

BATCH_SIZE = 8

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Saved model not found: {MODEL_DIR}"
    )

mapping_path = MODEL_DIR / "label_mapping.json"

if not mapping_path.exists():
    raise FileNotFoundError(
        f"Label mapping not found: {mapping_path}"
    )

with open(mapping_path, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

label2id = {
    str(label): int(idx)
    for label, idx in label_mapping["label2id"].items()
}

id2label = {
    int(idx): str(label)
    for idx, label in label_mapping["id2label"].items()
}

MAX_LENGTH = int(label_mapping["max_length"])

CLASS_NAMES = [
    id2label[i]
    for i in range(len(id2label))
]

print("\nClasses:", CLASS_NAMES)
print("Max length:", MAX_LENGTH)


if not os.path.exists(TEST_DATA_PATH):
    raise FileNotFoundError(TEST_DATA_PATH)

df = pd.read_csv(
    TEST_DATA_PATH,
    encoding="utf-8-sig",
    low_memory=False
)

if TEXT_COLUMN not in df.columns:
    raise ValueError(
        f"Column '{TEXT_COLUMN}' not found."
    )

if LABEL_COLUMN not in df.columns:
    raise ValueError(
        f"Column '{LABEL_COLUMN}' not found."
    )

df = df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()


df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)


df[LABEL_COLUMN] = (
    df[LABEL_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)


df = df[
    df[TEXT_COLUMN].ne("") &
    df[LABEL_COLUMN].notna() &
    df[LABEL_COLUMN].ne("")
].copy()


df = (
    df
    .drop_duplicates(subset=[TEXT_COLUMN])
    .reset_index(drop=True)
)

print("\nTotal evaluation samples:", len(df))

print("\nClass distribution:")
print(df[LABEL_COLUMN].value_counts())


test_df = df.copy().reset_index(drop=True)

texts = test_df[TEXT_COLUMN].tolist()

true_labels = (
    test_df[LABEL_COLUMN]
    .astype(str)
    .tolist()
)

print("\nAll available data will be evaluated.")
print("Evaluation samples:", len(test_df))


unknown_labels = (
    set(test_df[LABEL_COLUMN].astype(str).unique())
    - set(label2id.keys())
)

if unknown_labels:
    raise ValueError(
        f"Unknown labels in dataset: {unknown_labels}"
    )

y_true = np.array([
    label2id[label]
    for label in true_labels
])

print("\nLabels encoded successfully.")


print("\nLoading saved model...")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR)
)

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

model.to(device)
model.eval()

print("Model loaded successfully.")
print("Training will NOT be performed.")


all_probabilities = []

print("\nStarting prediction on the full dataset...")

with torch.inference_mode():

    for start in range(0, len(texts), BATCH_SIZE):

        end = min(
            start + BATCH_SIZE,
            len(texts)
        )

        batch_texts = texts[start:end]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        if device.type == "cuda":

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**encoded)

        else:
            outputs = model(**encoded)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=-1
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

        if end == len(texts) or end % 1000 == 0:
            print(
                f"Predicted: {end:,}/{len(texts):,}"
            )

        del encoded, outputs, probabilities

        if device.type == "cuda":
            torch.cuda.empty_cache()


all_probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

y_pred = np.argmax(
    all_probabilities,
    axis=1
)

predicted_labels = [
    id2label[int(idx)]
    for idx in y_pred
]

confidence = all_probabilities.max(axis=1)

correct = (y_true == y_pred)


results = pd.DataFrame({
    "text": texts,
    "true_label": true_labels,
    "predicted_label": predicted_labels,
    "confidence": confidence,
    "correct": correct
})

for class_id, class_name in id2label.items():

    results[f"prob_{class_name}"] = (
        all_probabilities[:, class_id]
    )


errors = results[
    results["correct"] == False
].copy()

errors = errors.sort_values(
    by="confidence",
    ascending=False
).reset_index(drop=True)


accuracy = accuracy_score(
    y_true,
    y_pred
)

print("\n" + "=" * 65)
print("FULL DATASET RESULTS")
print("=" * 65)

print("Total samples:", len(results))
print("Correct predictions:", int(correct.sum()))
print("Incorrect predictions:", len(errors))

print("Accuracy:", round(accuracy, 5))
print("Error rate:", round(1 - accuracy, 5))

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)


cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in CLASS_NAMES],
    columns=[f"pred_{x}" for x in CLASS_NAMES]
)

print("\nConfusion Matrix:")
print(cm_df)


print("\nError counts by true class:")

print(
    errors["true_label"].value_counts()
)

print("\nWrong prediction combinations:")

print(
    errors.groupby(
        ["true_label", "predicted_label"]
    ).size().sort_values(ascending=False)
)


print("\nTop 30 incorrect predictions:")

print(
    errors[
        [
            "text",
            "true_label",
            "predicted_label",
            "confidence"
        ]
    ].head(30).to_string(index=False)
)


print("\nTop 20 correct predictions:")

print(
    results[
        results["correct"] == True
    ][
        [
            "text",
            "true_label",
            "predicted_label",
            "confidence"
        ]
    ].head(20).to_string(index=False)
)


print("\n" + "=" * 65)
print("FULL DATASET ERROR ANALYSIS COMPLETED")
print("=" * 65)

print(f"Total samples: {len(results):,}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Correct: {int(correct.sum()):,}")
print(f"Incorrect: {len(errors):,}")

print("\nNo files were saved.")


del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

Classes: ['no_idea', 'not_recommended', 'recommended']
Max length: 128

Total evaluation samples: 10000

Class distribution:
recommendation_status
not_recommended    3600
recommended        3369
no_idea            3031
Name: count, dtype: Int64

All available data will be evaluated.
Evaluation samples: 10000

Labels encoded successfully.

Loading saved model...


W1113 05:47:17.938000 25396 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.
Training will NOT be performed.

Starting prediction on the full dataset...
Predicted: 1,000/10,000
Predicted: 2,000/10,000
Predicted: 3,000/10,000
Predicted: 4,000/10,000
Predicted: 5,000/10,000
Predicted: 6,000/10,000
Predicted: 7,000/10,000
Predicted: 8,000/10,000
Predicted: 9,000/10,000
Predicted: 10,000/10,000

FULL DATASET RESULTS
Total samples: 10000
Correct predictions: 7853
Incorrect predictions: 2147
Accuracy: 0.7853
Error rate: 0.2147

Classification Report:

                 precision    recall  f1-score   support

        no_idea     0.6242    0.7875    0.6964      3031
not_recommended     0.8487    0.7711    0.8080      3600
    recommended     0.9260    0.7985    0.8575      3369

       accuracy                         0.7853     10000
      macro avg     0.7996    0.7857    0.7873     10000
   weighted avg     0.8067    0.7853    0.7909     10000


Confusion Matrix:
                      pred_no_idea  pred_not_recommended  pred_recommended
tr